In [ ]:
import pandas as pd
import numpy as np

In [11]:
# charger dataset final (merge)
df = pd.read_csv(r"C:\Users\imane\Desktop\Fil-rouge\data\gold\f1_final_dataset.csv")

## 1. Calcul des KPI + Feature engineering

In [ ]:

# ==============================
# 1. FIX LOGIQUE Q/R
# ==============================

# En qualification → pas de finishPosition
df.loc[df["session"] == "Q", "finishPosition"] = np.nan

# En race → pas de qualyPosition direct (optionnel)
# df.loc[df["session"] == "R", "qualyPosition"] = np.nan

 # ==============================
# 2. FIX QUALITÉ (OPTIONNEL)
# ==============================
df = df[df.groupby(["season", "round", "session"])["driverCode"]
          .transform("count") >= 10]
 
# ============================================================
# FEATURE ENGINEERING + KPI CREATION
# ============================================================

df = df.copy()

# ------------------------------------------------------------
# 0) Colonnes numériques
# ------------------------------------------------------------
num_cols = [
    "bestLapTime_sec", "avgLapTime_sec", "stdLapTime_sec", "maxSpeed_kmh",
    "gridPosition", "finishPosition", "qualyPosition",
    "driverPoints", "constructorPoints", "estimatedSalaryUSD", "costCapUSD"
]

for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# ------------------------------------------------------------
# 1) KPI RACE GAIN
# Positions gagnées/perdues en course
# positif = gain, négatif = perte
# ------------------------------------------------------------

df["raceGain"] = np.where(
    (df["session"] == "R") &
    df["gridPosition"].notna() &
    df["finishPosition"].notna(),
    df["gridPosition"] - df["finishPosition"],
    np.nan
)

# ------------------------------------------------------------
# 2) KPI CONSISTENCY INDEX
# plus la valeur est faible, plus le pilote est régulier
# ------------------------------------------------------------
df["consistencyIndex"] = df["stdLapTime_sec"]

# ------------------------------------------------------------
# 3) KPI QUALIFYING PERFORMANCE INDEX
# pour les sessions Q : plus la position est petite, meilleure est la perf
# ------------------------------------------------------------
df["qualifyingPerformanceIndex"] = np.where(
    df["session"] == "Q",
    df["qualyPosition"],
    np.nan
)

# ------------------------------------------------------------
# 4) KPI TOP SPEED INDEX
# on reprend directement la vitesse max observée
# ------------------------------------------------------------
df["topSpeedIndex"] = df["maxSpeed_kmh"]

# ------------------------------------------------------------
# 5) KPI DRIVER ROI
# coût par point pilote = salaire / points
# plus faible = meilleur ROI
# ------------------------------------------------------------
df["driverROI"] = np.where(
    (df["driverPoints"].notna()) & (df["driverPoints"] > 0) &
    (df["estimatedSalaryUSD"].notna()),
    df["estimatedSalaryUSD"] / df["driverPoints"],
    np.nan
)

# ------------------------------------------------------------
# 6) KPI DRIVER EFFICIENCY
# points par million USD de salaire
# plus élevé = meilleur rendement
# ------------------------------------------------------------
df["driverEfficiency"] = np.where(
    (df["estimatedSalaryUSD"].notna()) & (df["estimatedSalaryUSD"] > 0),
    df["driverPoints"] / (df["estimatedSalaryUSD"] / 1_000_000),
    np.nan
)

# ------------------------------------------------------------
# 7) KPI BUDGET EFFICIENCY
# coût par point constructeur = budget / points
# plus faible = meilleur
# ------------------------------------------------------------
df["budgetEfficiency"] = np.where(
    (df["constructorPoints"].notna()) & (df["constructorPoints"] > 0) &
    (df["costCapUSD"].notna()),
    df["costCapUSD"] / df["constructorPoints"],
    np.nan
)

# ------------------------------------------------------------
# 8) KPI PERFORMANCE PER MILLION USD
# points constructeur par million USD investi
# plus élevé = meilleur
# ------------------------------------------------------------
df["pointsPerMillionUSD"] = np.where(
    (df["costCapUSD"].notna()) & (df["costCapUSD"] > 0),
    df["constructorPoints"] / (df["costCapUSD"] / 1_000_000),
    np.nan
)

# ------------------------------------------------------------
# 9) KPI SPEED EFFICIENCY
# vitesse max relative au budget
# ------------------------------------------------------------
df["speedEfficiency"] = np.where(
    (df["costCapUSD"].notna()) & (df["costCapUSD"] > 0),
    df["maxSpeed_kmh"] / (df["costCapUSD"] / 1_000_000),
    np.nan
)

# ------------------------------------------------------------
# 10) KPI COST PER PODIUM
# d'abord créer podium (course seulement)
# ------------------------------------------------------------
df["isPodium"] = np.where(
    (df["session"] == "R") & (df["finishPosition"].notna()) & (df["finishPosition"] <= 3),
    1,
    0
)

# nombre de podiums par saison/constructeur
podiums_team = (
    df[df["session"] == "R"]
    .groupby(["season", "constructorId"], as_index=False)["isPodium"]
    .sum()
    .rename(columns={"isPodium": "teamPodiums"})
)

df = df.merge(
    podiums_team,
    on=["season", "constructorId"],
    how="left"
)

df["costPerPodium"] = np.where(
    (df["teamPodiums"].notna()) & (df["teamPodiums"] > 0) &
    (df["costCapUSD"].notna()),
    df["costCapUSD"] / df["teamPodiums"],
    np.nan
)

# ------------------------------------------------------------
# 11) KPI TEAMMATE DELTA
# comparaison intra-équipe sur le best lap
# delta = temps pilote - meilleur temps de son coéquipier
# proche de 0 ou négatif = très bon
# ------------------------------------------------------------
team_best = df.groupby(
    ["season", "round", "session", "constructorId"]
)["bestLapTime_sec"].transform("min")

df["teammateDelta"] = df["bestLapTime_sec"] - team_best

# version moyenne par pilote/saison
df["avgTeammateDelta_season"] = df.groupby(
    ["season", "driverId"]
)["teammateDelta"].transform("mean")

# ------------------------------------------------------------
# 12) KPI PACE ADVANTAGE
# écart relatif au meilleur tour de la session
# plus faible = meilleur
# ------------------------------------------------------------
session_best_lap = df.groupby(
    ["season", "round", "session"]
)["bestLapTime_sec"].transform("min")

df["paceAdvantage"] = np.where(
    session_best_lap.notna() & (session_best_lap > 0),
    (df["bestLapTime_sec"] / session_best_lap) - 1,
    np.nan
)

# ------------------------------------------------------------
# 13) SPEED Z-SCORE
# vitesse relative par circuit
# ------------------------------------------------------------
df["speed_zscore"] = df.groupby("circuit")["maxSpeed_kmh"].transform(
    lambda x: (x - x.mean()) / x.std() if x.std() not in [0, np.nan] else np.nan
)

# ------------------------------------------------------------
# 14) POSITION RANK IN TEAM
# classement du pilote dans son équipe selon ses points
# ------------------------------------------------------------
df["pointsRankInTeam"] = df.groupby(
    ["season", "constructorId", "round"]
)["driverPoints"].rank(method="dense", ascending=False)

# ------------------------------------------------------------
# 15) QUALI TO RACE DELTA
# différence entre position qualif et arrivée
# positif = amélioration
# ------------------------------------------------------------
# df["qualiToRaceDelta"] = np.where(
#     (df["session"] == "R") &
#     df["gridPosition"].notna() &
#     df["finishPosition"].notna(),
#     df["gridPosition"] - df["finishPosition"],
#     np.nan
# )
df["qualiToRaceDelta"] = np.where(
    (df["session"] == "R") &
    df["qualyPosition"].notna() &
    df["finishPosition"].notna(),
    df["qualyPosition"] - df["finishPosition"],
    np.nan
)

# ------------------------------------------------------------
# 16) INDICATEURS AGRÉGÉS UTILES
# ------------------------------------------------------------

# moyenne finish position par pilote/saison
df["avgFinishPosition_season"] = df.groupby(
    ["season", "driverId"]
)["finishPosition"].transform("mean")

# moyenne qualy position par pilote/saison
df["avgQualyPosition_season"] = df.groupby(
    ["season", "driverId"]
)["qualyPosition"].transform("mean")

# moyenne race gain par pilote/saison
df["avgRaceGain_season"] = df.groupby(
    ["season", "driverId"]
)["raceGain"].transform("mean")

# moyenne consistency par pilote/saison
df["avgConsistency_season"] = df.groupby(
    ["season", "driverId"]
)["consistencyIndex"].transform("mean")

# moyenne speed par pilote/saison
df["avgTopSpeed_season"] = df.groupby(
    ["season", "driverId"]
)["topSpeedIndex"].transform("mean")

# ------------------------------------------------------------
# 17) SCORE PILOTE SIMPLE (optionnel)
# exemple composite
# ------------------------------------------------------------
# plus il est élevé, plus la performance globale est bonne
df["driverPerformanceScore"] = (
    df["avgRaceGain_season"].fillna(0) * 0.30
    - df["avgFinishPosition_season"].fillna(df["avgFinishPosition_season"].median()) * 0.20
    - df["avgQualyPosition_season"].fillna(df["avgQualyPosition_season"].median()) * 0.15
    - df["avgConsistency_season"].fillna(df["avgConsistency_season"].median()) * 0.10
    + df["driverEfficiency"].fillna(0) * 0.15
    + df["speed_zscore"].fillna(0) * 0.10
)

# ------------------------------------------------------------
# 18) APERÇU KPI
# ------------------------------------------------------------
kpi_cols = [
    "raceGain",
    "consistencyIndex",
    "qualifyingPerformanceIndex",
    "topSpeedIndex",
    "driverROI",
    "driverEfficiency",
    "budgetEfficiency",
    "pointsPerMillionUSD",
    "speedEfficiency",
    "isPodium",
    "teamPodiums",
    "costPerPodium",
    "teammateDelta",
    "avgTeammateDelta_season",
    "paceAdvantage",
    "speed_zscore",
    "pointsRankInTeam",
    "qualiToRaceDelta",
    "avgFinishPosition_season",
    "avgQualyPosition_season",
    "avgRaceGain_season",
    "avgConsistency_season",
    "avgTopSpeed_season",
    "driverPerformanceScore"
]

print("Colonnes KPI créées :")
print([c for c in kpi_cols if c in df.columns])

print("\nAperçu :")
print(df[kpi_cols].head())

# 19. AMÉLIORATIONS (AJOUT ICI)
# ==============================

# 🔧 1. Nettoyer driverROI extrêmes
df["driverROI"] = df["driverROI"].clip(
    upper=df["driverROI"].quantile(0.99)
)

# 🔧 2. Normalisation score (option ML)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df["driverPerformanceScore_scaled"] = scaler.fit_transform(
    df[["driverPerformanceScore"]]
)



Colonnes KPI créées :
['raceGain', 'consistencyIndex', 'qualifyingPerformanceIndex', 'topSpeedIndex', 'driverROI', 'driverEfficiency', 'budgetEfficiency', 'pointsPerMillionUSD', 'speedEfficiency', 'isPodium', 'teamPodiums', 'costPerPodium', 'teammateDelta', 'avgTeammateDelta_season', 'paceAdvantage', 'speed_zscore', 'pointsRankInTeam', 'qualiToRaceDelta', 'avgFinishPosition_season', 'avgQualyPosition_season', 'avgRaceGain_season', 'avgConsistency_season', 'avgTopSpeed_season', 'driverPerformanceScore']

Aperçu :
   raceGain  consistencyIndex  qualifyingPerformanceIndex  topSpeedIndex  \
0       NaN          0.583018                         9.0          314.0   
1     -10.0          0.506573                         NaN          319.0   
2       NaN          0.590492                         3.0          312.0   
3       0.0          1.100309                         NaN          297.0   
4       NaN          0.557317                         5.0          318.0   

       driverROI  driverE

### 1.2. Test de validation

In [ ]:
# ============================================================
# VALIDATION KPI + FEATURE ENGINEERING
# À exécuter après création des KPI sur df
# ============================================================


print("========================================")
print("VALIDATION KPI / FEATURE ENGINEERING")
print("========================================\n")

# ------------------------------------------------------------
# 0) Colonnes attendues
# ------------------------------------------------------------
expected_kpi_cols = [
    "raceGain",
    "consistencyIndex",
    "qualifyingPerformanceIndex",
    "topSpeedIndex",
    "driverROI",
    "driverEfficiency",
    "budgetEfficiency",
    "pointsPerMillionUSD",
    "speedEfficiency",
    "isPodium",
    "teamPodiums",
    "costPerPodium",
    "teammateDelta",
    "avgTeammateDelta_season",
    "paceAdvantage",
    "speed_zscore",
    "pointsRankInTeam",
    "qualiToRaceDelta",
    "avgFinishPosition_season",
    "avgQualyPosition_season",
    "avgRaceGain_season",
    "avgConsistency_season",
    "avgTopSpeed_season",
    "driverPerformanceScore"
]

missing_cols = [c for c in expected_kpi_cols if c not in df.columns]

print("1) Colonnes KPI manquantes :", missing_cols)
assert len(missing_cols) == 0, "❌ Certaines colonnes KPI n'ont pas été créées"

# ------------------------------------------------------------
# 1) raceGain
# ------------------------------------------------------------
# recalcul attendu
race_gain_check = df.loc[df["session"] == "R"].copy()
race_gain_check["expected"] = race_gain_check["gridPosition"] - race_gain_check["finishPosition"]

# comparaison seulement sur valeurs valides
mask = race_gain_check["expected"].notna()

race_gain_diff = np.sum(
    ~np.isclose(
        race_gain_check.loc[mask, "raceGain"],
        race_gain_check.loc[mask, "expected"],
        equal_nan=True
    )
)

print("Vérification raceGain incohérent :", race_gain_diff)
assert race_gain_diff == 0, "❌ raceGain incorrect"


# ------------------------------------------------------------
# 2) consistencyIndex = stdLapTime_sec
# ------------------------------------------------------------
consistency_diff = (df["consistencyIndex"] != df["stdLapTime_sec"]).sum()

print("3) Vérification consistencyIndex incohérent :", consistency_diff)
assert consistency_diff == 0, "❌ consistencyIndex incorrect"

# ------------------------------------------------------------
# 3) topSpeedIndex = maxSpeed_kmh
# ------------------------------------------------------------
topspeed_diff = (df["topSpeedIndex"] != df["maxSpeed_kmh"]).sum()

print("4) Vérification topSpeedIndex incohérent :", topspeed_diff)
assert topspeed_diff == 0, "❌ topSpeedIndex incorrect"

# ------------------------------------------------------------
# 4) isPodium cohérent
# ------------------------------------------------------------
podium_error = df[
    (df["session"] == "R") &
    (df["finishPosition"].notna()) &
    (
        ((df["finishPosition"] <= 3) & (df["isPodium"] != 1)) |
        ((df["finishPosition"] > 3) & (df["isPodium"] != 0))
    )
].shape[0]

print("5) Vérification isPodium incohérent :", podium_error)
assert podium_error == 0, "❌ isPodium incorrect"

# ------------------------------------------------------------
# 5) driverROI et driverEfficiency
# ------------------------------------------------------------
roi_negative = (df["driverROI"] < 0).sum()
eff_negative = (df["driverEfficiency"] < 0).sum()

print("6) driverROI négatif :", roi_negative)
print("7) driverEfficiency négatif :", eff_negative)

assert roi_negative == 0, "❌ driverROI négatif"
assert eff_negative == 0, "❌ driverEfficiency négatif"

# ------------------------------------------------------------
# 6) budgetEfficiency / pointsPerMillionUSD / speedEfficiency
# ------------------------------------------------------------
budget_eff_negative = (df["budgetEfficiency"] < 0).sum()
ppm_negative = (df["pointsPerMillionUSD"] < 0).sum()
speed_eff_negative = (df["speedEfficiency"] < 0).sum()

print("8) budgetEfficiency négatif :", budget_eff_negative)
print("9) pointsPerMillionUSD négatif :", ppm_negative)
print("10) speedEfficiency négatif :", speed_eff_negative)

assert budget_eff_negative == 0, "❌ budgetEfficiency négatif"
assert ppm_negative == 0, "❌ pointsPerMillionUSD négatif"
assert speed_eff_negative == 0, "❌ speedEfficiency négatif"

# ------------------------------------------------------------
# 7) paceAdvantage
# ------------------------------------------------------------
pace_invalid = (df["paceAdvantage"] < -0.01).sum()

print("11) paceAdvantage trop bas (< -0.01) :", pace_invalid)
assert pace_invalid == 0, "❌ paceAdvantage incohérent"

# ------------------------------------------------------------
# 8) pointsRankInTeam
# ------------------------------------------------------------
rank_invalid = df[
    df["pointsRankInTeam"].notna() &
    (df["pointsRankInTeam"] < 1)
].shape[0]

print("12) pointsRankInTeam invalide :", rank_invalid)
assert rank_invalid == 0, "❌ pointsRankInTeam incorrect"

# ------------------------------------------------------------
# 9) teamPodiums / costPerPodium
# ------------------------------------------------------------
team_podiums_negative = (df["teamPodiums"] < 0).sum()
cost_per_podium_negative = (df["costPerPodium"] < 0).sum()

print("13) teamPodiums négatif :", team_podiums_negative)
print("14) costPerPodium négatif :", cost_per_podium_negative)

assert team_podiums_negative == 0, "❌ teamPodiums négatif"
assert cost_per_podium_negative == 0, "❌ costPerPodium négatif"

# ------------------------------------------------------------
# 10) speed_zscore
# ------------------------------------------------------------
speed_zscore_all_nan = df["speed_zscore"].isna().all()

print("15) speed_zscore entièrement NaN :", speed_zscore_all_nan)
assert not speed_zscore_all_nan, "❌ speed_zscore non calculé"

# ------------------------------------------------------------
# 11) Qualifying / Race logique
# ------------------------------------------------------------
q_finish_error = df[(df["session"] == "Q") & (df["finishPosition"].notna())].shape[0]
r_finish_missing = df[(df["session"] == "R") & (df["finishPosition"].isna())].shape[0]

print("16) Q avec finishPosition :", q_finish_error)
print("17) R sans finishPosition :", r_finish_missing)

assert q_finish_error == 0, "❌ finishPosition ne doit pas exister en Q"
assert r_finish_missing == 0, "❌ finishPosition manquant en R"

# ------------------------------------------------------------
# 12) Résumé des KPI
# ------------------------------------------------------------
print("\nRésumé statistique des KPI principaux :")
print(df[
    [
        "raceGain",
        "driverROI",
        "driverEfficiency",
        "budgetEfficiency",
        "pointsPerMillionUSD",
        "speedEfficiency",
        "paceAdvantage",
        "driverPerformanceScore"
    ]
].describe())

# ------------------------------------------------------------
# TEST FINAL
# ------------------------------------------------------------
print("\n========================================")
print("✅ KPI / FEATURE ENGINEERING VALIDÉS")
print("========================================")

VALIDATION KPI / FEATURE ENGINEERING

1) Colonnes KPI manquantes : []
Vérification raceGain incohérent : 0
3) Vérification consistencyIndex incohérent : 0
4) Vérification topSpeedIndex incohérent : 0
5) Vérification isPodium incohérent : 0
6) driverROI négatif : 0
7) driverEfficiency négatif : 0
8) budgetEfficiency négatif : 0
9) pointsPerMillionUSD négatif : 0
10) speedEfficiency négatif : 0
11) paceAdvantage trop bas (< -0.01) : 0
12) pointsRankInTeam invalide : 0
13) teamPodiums négatif : 0
14) costPerPodium négatif : 0
15) speed_zscore entièrement NaN : False
16) Q avec finishPosition : 0
17) R sans finishPosition : 0

Résumé statistique des KPI principaux :
          raceGain     driverROI  driverEfficiency  budgetEfficiency  \
count  2037.000000  3.820000e+03       3973.000000      4.057000e+03   
mean      0.280314  2.001257e+05         14.639763      3.118954e+06   
std       4.737248  2.839692e+05         15.774784      5.704477e+06   
min     -19.000000  1.333333e+04         

### 1.3. Exportation data finale

In [14]:
from pathlib import Path
PROJECT_DIR = Path(r"C:\Users\imane\Desktop\Fil-rouge")
GOLD_DIR = PROJECT_DIR / "data" / "gold"
GOLD_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = GOLD_DIR / "f1_final_enriched.csv"

df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print("✅ Dataset enrichi exporté")
print("📁 Fichier :", OUTPUT_PATH)
print("Shape :", df.shape)

✅ Dataset enrichi exporté
📁 Fichier : C:\Users\imane\Desktop\Fil-rouge\data\gold\f1_final_enriched.csv
Shape : (4129, 54)
